# 01 — Exploratory Data Analysis

IMDB Large Movie Review Dataset (50,000 reviews, binary sentiment).

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np

dataset = load_dataset('stanfordnlp/imdb')
print(dataset)

In [ ]:
train = dataset['train']
test  = dataset['test']

print(f"Train size: {len(train)}")
print(f"Test size:  {len(test)}")

labels = train['label']
neg = labels.count(0)
pos = labels.count(1)
print(f"Train — negative: {neg} | positive: {pos} (balanced: {neg==pos})")

In [ ]:
# Review length distribution
lengths = [len(r.split()) for r in train['text']]

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(lengths, bins=60, color='steelblue', edgecolor='white')
ax.axvline(np.median(lengths), color='red', linestyle='--', label=f'median={int(np.median(lengths))}')
ax.set_xlabel('words per review')
ax.set_ylabel('count')
ax.set_title('Review length distribution (train)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/eda_length_dist.png', dpi=120)
plt.show()

print(f"Min: {min(lengths)} | Max: {max(lengths)} | Median: {int(np.median(lengths))} | Mean: {int(np.mean(lengths))}")

In [ ]:
# Token count after DistilBERT tokenization
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')
sample = train.select(range(2000))
token_lengths = [len(tokenizer(r, truncation=False)['input_ids']) for r in sample['text']]

pct_over_512 = sum(t > 512 for t in token_lengths) / len(token_lengths) * 100
print(f"Reviews with >512 tokens (truncated): {pct_over_512:.1f}%")
print(f"Median token count: {int(np.median(token_lengths))}")

In [ ]:
# Sample reviews
for label_id, label_name in [(0, 'negative'), (1, 'positive')]:
    examples = [r for r, l in zip(train['text'], train['label']) if l == label_id][:2]
    print(f"--- {label_name.upper()} ---")
    for e in examples:
        print(e[:300], '...\n')